# ATML PA1 -- Colab bootstrap

Run this once per session. It mounts Drive (for datasets/checkpoints, which are git-ignored
and must survive a disconnect), clones/pulls this repo (the source of truth for code and
directory structure), and installs dependencies. After this, run everything as
`!python -m task2.train --config task2/configs/dann.yaml`-style commands from the repo root
so the exact same scripts work locally too.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/atml_pa1'
DATA_ROOT = f'{DRIVE_ROOT}/data'       # STL-10 / PACS / CIFAR-10 / CIFAR-100 cache
CKPT_ROOT = f'{DRIVE_ROOT}/checkpoints' # everything task*/results/*/checkpoint.pt points at

import os
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(CKPT_ROOT, exist_ok=True)
print('Data root:', DATA_ROOT)
print('Checkpoint root:', CKPT_ROOT)

In [ ]:
REPO_URL = 'https://github.com/adilawan1/ATML-PA1.git'
REPO_DIR = '/content/ATML-PA1'

if os.path.isdir(REPO_DIR):
    %cd $REPO_DIR
    !git pull
else:
    !git clone $REPO_URL $REPO_DIR
    %cd $REPO_DIR

In [ ]:
# IMPORTANT: do NOT `pip install torch`/`torchvision` here. Colab preinstalls a build of
# each already matched to its GPU driver; PyPI's default (untagged) wheel for both is
# CPU-only, and reinstalling them is exactly what causes
# "AssertionError: Torch not compiled with CUDA enabled" later on. Install everything else
# from requirements.txt and leave those two alone.
!grep -vE '^(torch|torchvision)$' requirements.txt > /tmp/requirements_colab.txt
!pip install -q -r /tmp/requirements_colab.txt

# Sanity check -- if this ever prints False/None on a GPU runtime, something (re)installed a
# CPU-only torch; fix with:
#   !pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
import torch
print("torch", torch.__version__, "| cuda build:", torch.version.cuda, "| available:", torch.cuda.is_available())

In [ ]:
!nvidia-smi

## Task 4 kickoff (Vanilla + GCSC)

These are the two required Task 4 trainings that need no manual dataset download (CIFAR-10
fetches automatically) and don't depend on PACS. Each cell checks for an existing checkpoint
first, so it's safe to re-run this section if the runtime disconnects mid-way -- it will skip
anything already finished rather than retraining from scratch. Run the three cells below in
order, then the commit cell once both are done.

In [ ]:
import os

SPLIT_PATH = "task4/data/cifar10_split_seed6304.json"
if not os.path.exists(SPLIT_PATH):
    !python -m task4.data.make_splits --data-root {DATA_ROOT}
else:
    print(f"{SPLIT_PATH} already exists, skipping.")

In [ ]:
import os

from task4.methods.vanilla import train_vanilla

VANILLA_CKPT = f"{CKPT_ROOT}/task4/vanilla/checkpoint.pt"
if os.path.exists(VANILLA_CKPT):
    print(f"Found existing checkpoint at {VANILLA_CKPT}, skipping training. Delete it to retrain.")
else:
    vanilla_model = train_vanilla(
        data_root=DATA_ROOT,
        split_path="task4/data/cifar10_split_seed6304.json",
        device="cuda",
        checkpoint_path=VANILLA_CKPT,
        metrics_path="task4/results/vanilla/metrics.jsonl",
    )

In [ ]:
import os

from task4.methods.gcsc import train_gcsc

GCSC_CKPT = f"{CKPT_ROOT}/task4/gcsc/checkpoint.pt"
if os.path.exists(GCSC_CKPT):
    print(f"Found existing checkpoint at {GCSC_CKPT}, skipping training. Delete it to retrain.")
else:
    gcsc_model = train_gcsc(
        data_root=DATA_ROOT,
        split_path="task4/data/cifar10_split_seed6304.json",
        device="cuda",
        checkpoint_path=GCSC_CKPT,
        metrics_path="task4/results/gcsc/metrics.jsonl",
    )

In [ ]:
!git add task4/data/cifar10_split_seed6304.json task4/results/vanilla/metrics.jsonl task4/results/gcsc/metrics.jsonl
!git commit -m "Add Task 4 Vanilla and GCSC training curves"
!git push

## Running a task

Point every `--data-root` / `pacs_root` at `DATA_ROOT`, and every config's `output.checkpoint`
at a path under `CKPT_ROOT` (edit the YAML, or symlink `task2/results` etc. into
`CKPT_ROOT` -- either works, just be consistent so a disconnect doesn't lose a checkpoint).

```python
!python -m shared.pacs_protocol --root {DATA_ROOT}/pacs
!python -m task2.train --config task2/configs/source_only.yaml
```

After any run that produced new small result files (JSON/CSV/figures, not checkpoints),
commit and push from a cell:

```python
!git add task2/results/*.json report/figures
!git commit -m "Add Task 2 source-only results"
!git push
```